### 다중 벡터저장소 검색기 ( MultiVectorRetriever )

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_teddynote import logging

logging.langsmith("test0914")

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914


In [ ]:
%pip install pymupdf

In [5]:
# 텍스트파일에서 데이터를 로드하고, 로드된 문서들을 지정된 크기로 분할하는 전처리 과정

from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("SPRI_AI_Brief_2023년12월호_F.pdf")
docs = loader.load()

In [6]:
# 데이터로부터 로드한 원본 도큐먼트를 docs 변수에 담음
print(docs[5].page_content[:500])

1. 정책/법제  
2. 기업/산업 
3. 기술/연구 
 4. 인력/교육
영국 AI 안전성 정상회의에 참가한 28개국, AI 위험에 공동 대응 선언
n 영국 블레츨리 파크에서 개최된 AI 안전성 정상회의에 참가한 28개국들이 AI 안전 보장을 
위한 협력 방안을 담은 블레츨리 선언을 발표
n 첨단 AI를 개발하는 국가와 기업들은 AI 시스템에 대한 안전 테스트 계획에 합의했으며, 
영국의 AI 안전 연구소가 전 세계 국가와 협력해 테스트를 주도할 예정 
KEY Contents
£ AI 안전성 정상회의 참가국들, 블레츨리 선언 통해 AI 안전 보장을 위한 협력에 합의
n 2023년 11월 1~2일 영국 블레츨리 파크에서 열린 AI 안전성 정상회의(AI Safety Summit)에 
참가한 28개국 대표들이 AI 위험 관리를 위한 ‘블레츨리 선언’을 발표 
∙선언은 AI 안전 보장을 위해 국가, 국제기구, 기업, 시민사회, 학계를 포함한 모든 이해관계자의 협력이 
중요하다고 강조했으며,


- Chunk + 원본 문서 검색
- 대용량 정보를 검색하는 경우, 더 작은 단위로 정보를 임베딩하는 것이 유용할 수 있음
- MultiVectorRetriever 를 통해 문서를 여러 벡터로 저장하고 관리할 수 있음
- docstore에 원본 문서를 저장하고, vectorstore에 임베딩된 문서를 저장

In [7]:
import uuid
from langchain_classic.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever

vectorstore = Chroma(
    collection_name="small_bigger_chunks",
    embedding_function= OpenAIEmbeddings(model="text-embedding-3-small")
)

# 부모 문서의 저장소 계층
store = InMemoryStore()

id_key = "doc_id"

# 검색기 ( 시작 시 비어 있음 )
retriever = MultiVectorRetriever(
    vectorstore= vectorstore,
    byte_store= store,
    id_key= id_key
)

# 문서 ID를 생성합니다.
doc_ids = [str(uuid.uuid4()) for _ in docs]

# 두개의 생성된 id를 확인
doc_ids

['3927b3ab-8999-41da-b387-9832f58176c0',
 '7fd140e4-5d44-4771-a1b2-fc7ba2c87625',
 'afcc79fa-6fd8-4457-b9ec-d16d0b6361e6',
 'ec914fd9-b863-4bc1-b05c-621c7a4190b3',
 '9ce28fe1-30bb-46f6-b9ba-5a23b43746fb',
 '38cba1dd-2dc6-43c5-a3de-e9c5d2c94716',
 'c394f43b-714e-4791-abb1-5cd4a2d8b14e',
 'f0a1f7d9-a9cd-4230-9387-5f61c3b920dd',
 '62a85e7d-e53b-4a1c-92b8-b3ae2aecc71c',
 '385e27d3-20cb-46a2-9547-ff896eecc6d5',
 '8d5acab5-e661-47c3-80dd-0ebdb1c8ea71',
 'be5df744-5f5c-4497-a64a-d118e7584cb4',
 '2e36f9bd-6434-4632-8fc5-d74be1e4dffa',
 '2bf78343-c490-4a70-a4a1-d3e9dca229fb',
 'ed5264ff-15e6-449a-9b7c-ec854b539ca8',
 '8158978c-a176-404e-8e02-d525401b5b83',
 '5df10ef7-096d-486f-abd5-9d53c01e11ee',
 'b78ab714-7b94-4d74-b5c4-2541934341da',
 '44e68f2f-9650-492e-8410-d0890ca1ed98',
 '7224c78d-8004-48ab-9143-0ad727f03f17',
 'e440c093-48bb-479c-abef-c7e4e88ee152',
 '030d3d1a-7250-4b7c-9e73-58fe60653210',
 'b8490edf-0b2c-46da-969c-d355f3b69337']

In [8]:
parent_text_splitter = RecursiveCharacterTextSplitter(chunk_size = 600)

child_text_splitter = RecursiveCharacterTextSplitter(chunk_size = 200)

In [9]:
parent_docs = []

for i, doc in enumerate(docs):
    # 현재 문서의 ID를 가져옵니다.
    _id = doc_ids[i]
    # 현재 문서를 하위 문서로 분할
    parent_doc = parent_text_splitter.split_documents([doc])

    for _doc in parent_doc:
        # metadata에 문서 ID를 저장
        _doc.metadata[id_key] = _id
    parent_docs.extend(parent_doc)

In [10]:
# 생성된 parent 문서의 메타데이터 확인
parent_docs[0].metadata

{'producer': 'Hancom PDF 1.3.0.542',
 'creator': 'Hwp 2018 10.0.0.13462',
 'creationdate': '2023-12-08T13:28:38+09:00',
 'source': 'SPRI_AI_Brief_2023년12월호_F.pdf',
 'file_path': 'SPRI_AI_Brief_2023년12월호_F.pdf',
 'total_pages': 23,
 'format': 'PDF 1.4',
 'title': '',
 'author': 'dj',
 'subject': '',
 'keywords': '',
 'moddate': '2023-12-08T13:28:38+09:00',
 'trapped': '',
 'modDate': "D:20231208132838+09'00'",
 'creationDate': "D:20231208132838+09'00'",
 'page': 0,
 'doc_id': '3927b3ab-8999-41da-b387-9832f58176c0'}

In [11]:
child_docs = []

for i, doc in enumerate(docs):
    _id = doc_ids[i]

    child_doc = child_text_splitter.split_documents([doc])

    for _doc in child_doc:
        _doc.metadata[id_key] = _id
    child_docs.extend(child_doc)

In [12]:
child_docs[0].metadata

{'producer': 'Hancom PDF 1.3.0.542',
 'creator': 'Hwp 2018 10.0.0.13462',
 'creationdate': '2023-12-08T13:28:38+09:00',
 'source': 'SPRI_AI_Brief_2023년12월호_F.pdf',
 'file_path': 'SPRI_AI_Brief_2023년12월호_F.pdf',
 'total_pages': 23,
 'format': 'PDF 1.4',
 'title': '',
 'author': 'dj',
 'subject': '',
 'keywords': '',
 'moddate': '2023-12-08T13:28:38+09:00',
 'trapped': '',
 'modDate': "D:20231208132838+09'00'",
 'creationDate': "D:20231208132838+09'00'",
 'page': 0,
 'doc_id': '3927b3ab-8999-41da-b387-9832f58176c0'}

In [13]:
# 각각 분할된 청크의 수 확인
print(f"분할된 parent_docs의 개수 : {len(parent_docs)}")
print(f"분할된 child_docs의 개수 : {len(child_docs)}")

분할된 parent_docs의 개수 : 73
분할된 child_docs의 개수 : 440


In [15]:
# 벡터 저장소에 parent + child 문서를 추가
retriever.vectorstore.add_documents(parent_docs)
retriever.vectorstore.add_documents(child_docs)

retriever.docstore.mset(list(zip(doc_ids, docs)))

In [16]:
# vectorstore의 유사도 검색을 수행
relevant_chunks = retriever.vectorstore.similarity_search(
    "삼성전자가 만든 생성형 AI의 이름은?"
)
print(f"검색된 문서의 개수: {len(relevant_chunks)}")

검색된 문서의 개수: 4


In [17]:
for chunk in relevant_chunks:
    print(chunk.page_content, end="\n\n")
    print(">" * 100, end="\n\n")

☞ 출처 : 삼성전자, ‘삼성 AI 포럼’서 자체 개발 생성형 AI ‘삼성 가우스’ 공개, 2023.11.08.
삼성전자, ‘삼성 개발자 콘퍼런스 코리아 2023’ 개최, 2023.11.14.
TechRepublic, Samsung Gauss: Samsung Research Reveals Generative AI, 2023.11.08.

>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

☞ 출처 : 삼성전자, ‘삼성 AI 포럼’서 자체 개발 생성형 AI ‘삼성 가우스’ 공개, 2023.11.08.
삼성전자, ‘삼성 개발자 콘퍼런스 코리아 2023’ 개최, 2023.11.14.
TechRepublic, Samsung Gauss: Samsung Research Reveals Generative AI, 2023.11.08.

>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

SPRi AI Brief |  
2023-12월호
10
삼성전자, 자체 개발 생성 AI ‘삼성 가우스’ 공개
n 삼성전자가 온디바이스에서 작동 가능하며 언어, 코드, 이미지의 3개 모델로 구성된 자체 개발 생성 
AI 모델 ‘삼성 가우스’를 공개
n 삼성전자는 삼성 가우스를 다양한 제품에 단계적으로 탑재할 계획으로, 온디바이스 작동이 가능한

>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

SPRi AI Brief |  
2023-12월호
10
삼성전자, 자체 개발 생성 AI ‘삼성 가우스’ 공개
n 삼성전자가 온디바이스에서 작동 가능하며 언어, 코드, 이미지의 3개 모델로 구성된 자체 개발 생성 
AI

In [20]:
# retriever.invoke() 메서드를 사용하여 쿼리를 실행
relevant_docs = retriever.invoke("삼성전자가 만든 생성형 AI의 이름은 ?")
print(f"검색된 문서의 개수: {len(relevant_docs)}", end="\n\n")
print("=" * 100, end="\n\n")
print(relevant_docs[0].page_content)

검색된 문서의 개수: 1


SPRi AI Brief |  
2023-12월호
10
삼성전자, 자체 개발 생성 AI ‘삼성 가우스’ 공개
n 삼성전자가 온디바이스에서 작동 가능하며 언어, 코드, 이미지의 3개 모델로 구성된 자체 개발 생성 
AI 모델 ‘삼성 가우스’를 공개
n 삼성전자는 삼성 가우스를 다양한 제품에 단계적으로 탑재할 계획으로, 온디바이스 작동이 가능한 
삼성 가우스는 외부로 사용자 정보가 유출될 위험이 없다는 장점을 보유
KEY Contents
£ 언어, 코드, 이미지의 3개 모델로 구성된 삼성 가우스, 온디바이스 작동 지원
n 삼성전자가 2023년 11월 8일 열린 ‘삼성 AI 포럼 2023’ 행사에서 자체 개발한 생성 AI 모델 
‘삼성 가우스’를 최초 공개
∙정규분포 이론을 정립한 천재 수학자 가우스(Gauss)의 이름을 본뜬 삼성 가우스는 다양한 상황에 
최적화된 크기의 모델 선택이 가능
∙삼성 가우스는 라이선스나 개인정보를 침해하지 않는 안전한 데이터를 통해 학습되었으며, 
온디바이스에서 작동하도록 설계되어 외부로 사용자의 정보가 유출되지 않는 장점을 보유
∙삼성전자는 삼성 가우스를 활용한 온디바이스 AI 기술도 소개했으며, 생성 AI 모델을 다양한 제품에 
단계적으로 탑재할 계획
n 삼성 가우스는 △텍스트를 생성하는 언어모델 △코드를 생성하는 코드 모델 △이미지를 생성하는 
이미지 모델의 3개 모델로 구성
∙언어 모델은 클라우드와 온디바이스 대상 다양한 모델로 구성되며, 메일 작성, 문서 요약, 번역 업무의 
처리를 지원
∙코드 모델 기반의 AI 코딩 어시스턴트 ‘코드아이(code.i)’는 대화형 인터페이스로 서비스를 제공하며 
사내 소프트웨어 개발에 최적화
∙이미지 모델은 창의적인 이미지를 생성하고 기존 이미지를 원하는 대로 바꿀 수 있도록 지원하며 
저해상도 이미지의 고해상도 전환도 지원
n IT 전문지 테크리퍼블릭(TechRepublic)은 온디바이스 AI가 주요 기술 트렌드로 부상했다며, 
2024년부터 가우스를 탑재한 삼성

- retriever가 벡터 데이터베이스에서 기본적으로 수행하는 검색 유형은 유사도 검색

In [21]:
from langchain_classic.retrievers.multi_vector import SearchType

# 검색 유형을 MMR(Maximal Marginal Relevance)로 설정
retriever.search_type = SearchType.mmr

# 관련 문서 전체를 검색
print(retriever.invoke("삼성전자가 만든 생성형 AI의 이름은?")[0].page_content)

SPRi AI Brief |  
2023-12월호
10
삼성전자, 자체 개발 생성 AI ‘삼성 가우스’ 공개
n 삼성전자가 온디바이스에서 작동 가능하며 언어, 코드, 이미지의 3개 모델로 구성된 자체 개발 생성 
AI 모델 ‘삼성 가우스’를 공개
n 삼성전자는 삼성 가우스를 다양한 제품에 단계적으로 탑재할 계획으로, 온디바이스 작동이 가능한 
삼성 가우스는 외부로 사용자 정보가 유출될 위험이 없다는 장점을 보유
KEY Contents
£ 언어, 코드, 이미지의 3개 모델로 구성된 삼성 가우스, 온디바이스 작동 지원
n 삼성전자가 2023년 11월 8일 열린 ‘삼성 AI 포럼 2023’ 행사에서 자체 개발한 생성 AI 모델 
‘삼성 가우스’를 최초 공개
∙정규분포 이론을 정립한 천재 수학자 가우스(Gauss)의 이름을 본뜬 삼성 가우스는 다양한 상황에 
최적화된 크기의 모델 선택이 가능
∙삼성 가우스는 라이선스나 개인정보를 침해하지 않는 안전한 데이터를 통해 학습되었으며, 
온디바이스에서 작동하도록 설계되어 외부로 사용자의 정보가 유출되지 않는 장점을 보유
∙삼성전자는 삼성 가우스를 활용한 온디바이스 AI 기술도 소개했으며, 생성 AI 모델을 다양한 제품에 
단계적으로 탑재할 계획
n 삼성 가우스는 △텍스트를 생성하는 언어모델 △코드를 생성하는 코드 모델 △이미지를 생성하는 
이미지 모델의 3개 모델로 구성
∙언어 모델은 클라우드와 온디바이스 대상 다양한 모델로 구성되며, 메일 작성, 문서 요약, 번역 업무의 
처리를 지원
∙코드 모델 기반의 AI 코딩 어시스턴트 ‘코드아이(code.i)’는 대화형 인터페이스로 서비스를 제공하며 
사내 소프트웨어 개발에 최적화
∙이미지 모델은 창의적인 이미지를 생성하고 기존 이미지를 원하는 대로 바꿀 수 있도록 지원하며 
저해상도 이미지의 고해상도 전환도 지원
n IT 전문지 테크리퍼블릭(TechRepublic)은 온디바이스 AI가 주요 기술 트렌드로 부상했다며, 
2024년부터 가우스를 탑재한 삼성 스마트폰이 메타의 라마(Ll

In [22]:
# 검색 유형을 similarity_score_threshold로 설정
retriever.search_type = SearchType.similarity_score_threshold
retriever.search_kwargs = {"score_threshold" : 0.3}

# 관련 문서 전체를 검색
print(retriever.invoke("삼성전자가 만든 생성형 AI의 이름은?")[0].page_content)

SPRi AI Brief |  
2023-12월호
10
삼성전자, 자체 개발 생성 AI ‘삼성 가우스’ 공개
n 삼성전자가 온디바이스에서 작동 가능하며 언어, 코드, 이미지의 3개 모델로 구성된 자체 개발 생성 
AI 모델 ‘삼성 가우스’를 공개
n 삼성전자는 삼성 가우스를 다양한 제품에 단계적으로 탑재할 계획으로, 온디바이스 작동이 가능한 
삼성 가우스는 외부로 사용자 정보가 유출될 위험이 없다는 장점을 보유
KEY Contents
£ 언어, 코드, 이미지의 3개 모델로 구성된 삼성 가우스, 온디바이스 작동 지원
n 삼성전자가 2023년 11월 8일 열린 ‘삼성 AI 포럼 2023’ 행사에서 자체 개발한 생성 AI 모델 
‘삼성 가우스’를 최초 공개
∙정규분포 이론을 정립한 천재 수학자 가우스(Gauss)의 이름을 본뜬 삼성 가우스는 다양한 상황에 
최적화된 크기의 모델 선택이 가능
∙삼성 가우스는 라이선스나 개인정보를 침해하지 않는 안전한 데이터를 통해 학습되었으며, 
온디바이스에서 작동하도록 설계되어 외부로 사용자의 정보가 유출되지 않는 장점을 보유
∙삼성전자는 삼성 가우스를 활용한 온디바이스 AI 기술도 소개했으며, 생성 AI 모델을 다양한 제품에 
단계적으로 탑재할 계획
n 삼성 가우스는 △텍스트를 생성하는 언어모델 △코드를 생성하는 코드 모델 △이미지를 생성하는 
이미지 모델의 3개 모델로 구성
∙언어 모델은 클라우드와 온디바이스 대상 다양한 모델로 구성되며, 메일 작성, 문서 요약, 번역 업무의 
처리를 지원
∙코드 모델 기반의 AI 코딩 어시스턴트 ‘코드아이(code.i)’는 대화형 인터페이스로 서비스를 제공하며 
사내 소프트웨어 개발에 최적화
∙이미지 모델은 창의적인 이미지를 생성하고 기존 이미지를 원하는 대로 바꿀 수 있도록 지원하며 
저해상도 이미지의 고해상도 전환도 지원
n IT 전문지 테크리퍼블릭(TechRepublic)은 온디바이스 AI가 주요 기술 트렌드로 부상했다며, 
2024년부터 가우스를 탑재한 삼성 스마트폰이 메타의 라마(Ll

In [23]:
# 검색 유형을 similarity로 설정, k 값을 1로 설정
retriever.search_type = SearchType.similarity
retriever.search_kwargs = {"k" : 1}

# 관련 문서 전체를 검색
print(len(retriever.invoke("삼성전자가 만든 생성형 AI 의 이름은?")))

1


- 요약본을 벡터저장소에 저장
- 요약은 종종 청크의 냉을 보다 정확하게 추출할 수 있어 더 나은 검색 결과를 얻을 수 있음

In [24]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# pdf파일 로더 초기화
loader = PyMuPDFLoader("SPRI_AI_Brief_2023년12월호_F.pdf")

# 텍스트 분할
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 600, chunk_overlap = 50)

# PDF 파일 로드 및 텍스트 분할 실행
split_docs = loader.load_and_split(text_splitter)

# 분할된 문서의 개수 출력
print(f"분할된 문서의 개수: {len(split_docs)}")

분할된 문서의 개수: 61


In [26]:
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

summary_chain = (
    {"doc" : lambda x: x.page_content}
    | ChatPromptTemplate.from_messages(
        [
            ("system", "You are an expert in summarizing documents in Korean."),
            ("user", "Summarize the following documents in 3 sentences in bullet points format.\n\n{doc}")
        ]
    )
    | ChatOpenAI(temperature=0, model="gpt-4o-mini") | StrOutputParser()
)

- chain.batch 메서드를 사용하여 docs 리스트의 문서들을 일괄 요약
- max_concurrency 매개변수를 10으로 설정하여 최대 10개의 문서를 동시에 처리

In [27]:
summaries = summary_chain.batch(split_docs, {"max_concurrency" : 10})

In [ ]:
len(summaries) # 분할한 문서의 개수와 동일한 61개

61

In [29]:
# 원본 문서의 내용 출력
print(split_docs[33].page_content, end="\n\n")

# 요약 출력
print("[요약]")
print(summaries[33])

SPRi AI Brief |  
2023-12월호
10
삼성전자, 자체 개발 생성 AI ‘삼성 가우스’ 공개
n 삼성전자가 온디바이스에서 작동 가능하며 언어, 코드, 이미지의 3개 모델로 구성된 자체 개발 생성 
AI 모델 ‘삼성 가우스’를 공개
n 삼성전자는 삼성 가우스를 다양한 제품에 단계적으로 탑재할 계획으로, 온디바이스 작동이 가능한 
삼성 가우스는 외부로 사용자 정보가 유출될 위험이 없다는 장점을 보유
KEY Contents
£ 언어, 코드, 이미지의 3개 모델로 구성된 삼성 가우스, 온디바이스 작동 지원
n 삼성전자가 2023년 11월 8일 열린 ‘삼성 AI 포럼 2023’ 행사에서 자체 개발한 생성 AI 모델 
‘삼성 가우스’를 최초 공개
∙정규분포 이론을 정립한 천재 수학자 가우스(Gauss)의 이름을 본뜬 삼성 가우스는 다양한 상황에 
최적화된 크기의 모델 선택이 가능
∙삼성 가우스는 라이선스나 개인정보를 침해하지 않는 안전한 데이터를 통해 학습되었으며, 
온디바이스에서 작동하도록 설계되어 외부로 사용자의 정보가 유출되지 않는 장점을 보유
∙삼성전자는 삼성 가우스를 활용한 온디바이스 AI 기술도 소개했으며, 생성 AI 모델을 다양한 제품에

[요약]
- 삼성전자가 자체 개발한 생성 AI 모델 '삼성 가우스'를 공개하였으며, 이 모델은 언어, 코드, 이미지의 3개 모델로 구성되어 온디바이스에서 작동 가능하다.  
- '삼성 가우스'는 정규분포 이론을 정립한 수학자 가우스의 이름을 따왔으며, 다양한 상황에 최적화된 모델 선택이 가능하다.  
- 삼성전자는 이 AI 모델이 사용자 정보를 외부로 유출하지 않도록 설계되었으며, 향후 다양한 제품에 단계적으로 탑재할 계획이다.  


In [30]:
import uuid

# 요약 정보를 저장할 벡터 저장소를 생성
summary_vectorstore = Chroma(
    collection_name="summaries",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small")
)

# 부모 문서를 저장할 저장소 생성
store = InMemoryStore()

# 문서 ID를 저장할 키 이름을 지정
id_key = "doc_id"

# 검색기를 초기화( 시작 시 비어 있음 )
retriever = MultiVectorRetriever(
    vectorstore= summary_vectorstore,
    byte_store= store,
    id_key= id_key
)

# 문서 ID를 생성
doc_ids = [str(uuid.uuid4()) for _ in split_docs]

In [31]:
summary_docs = [
    # 요약된 내용을 페이지 콘텐츠로 하고, 문서 ID를 메타데이터로 포함하는 Document 객체를 생성
    Document(page_content=s, metadata={id_key: doc_ids[i]})
    for i,s in enumerate(summaries)
]

In [32]:
# 요약본의 문서의 개수
len(summary_docs)

61

In [34]:
# 요약된 문서를 벡터 저장소에 추가
retriever.vectorstore.add_documents(summary_docs)

# 문서 ID와 문서를 매핑하여 문서 저장소에 저장
retriever.docstore.mset(list(zip(doc_ids, split_docs)))

In [35]:
# 유사도 검색 수행
result_docs = summary_vectorstore.similarity_search("삼성전자가 만든 생성형 AI의 이름은 ?")

In [40]:
print(result_docs[0].page_content)

- 삼성전자가 자체 개발한 생성 AI 모델 '삼성 가우스'를 공개하였으며, 이 모델은 언어, 코드, 이미지의 3개 모델로 구성되어 온디바이스에서 작동 가능하다.  
- '삼성 가우스'는 정규분포 이론을 정립한 수학자 가우스의 이름을 따왔으며, 다양한 상황에 최적화된 모델 선택이 가능하다.  
- 삼성전자는 이 AI 모델이 사용자 정보를 외부로 유출하지 않도록 설계되었으며, 향후 다양한 제품에 단계적으로 탑재할 계획이다.  


In [41]:
# 관련된 문서를 검색하여 가져옴
retrieved_docs = retriever.invoke("삼성전자가 만든 생성형 AI의 이름은?")
print(retrieved_docs[0].page_content)

SPRi AI Brief |  
2023-12월호
10
삼성전자, 자체 개발 생성 AI ‘삼성 가우스’ 공개
n 삼성전자가 온디바이스에서 작동 가능하며 언어, 코드, 이미지의 3개 모델로 구성된 자체 개발 생성 
AI 모델 ‘삼성 가우스’를 공개
n 삼성전자는 삼성 가우스를 다양한 제품에 단계적으로 탑재할 계획으로, 온디바이스 작동이 가능한 
삼성 가우스는 외부로 사용자 정보가 유출될 위험이 없다는 장점을 보유
KEY Contents
£ 언어, 코드, 이미지의 3개 모델로 구성된 삼성 가우스, 온디바이스 작동 지원
n 삼성전자가 2023년 11월 8일 열린 ‘삼성 AI 포럼 2023’ 행사에서 자체 개발한 생성 AI 모델 
‘삼성 가우스’를 최초 공개
∙정규분포 이론을 정립한 천재 수학자 가우스(Gauss)의 이름을 본뜬 삼성 가우스는 다양한 상황에 
최적화된 크기의 모델 선택이 가능
∙삼성 가우스는 라이선스나 개인정보를 침해하지 않는 안전한 데이터를 통해 학습되었으며, 
온디바이스에서 작동하도록 설계되어 외부로 사용자의 정보가 유출되지 않는 장점을 보유
∙삼성전자는 삼성 가우스를 활용한 온디바이스 AI 기술도 소개했으며, 생성 AI 모델을 다양한 제품에


### 가설 쿼리 ( Hypothetical Queries )를 활용하여 문서 내용 탐색

In [44]:
functions = [
    {
        "name" : "hypothetical_question", # 함수의 이름을 지정
        "description" : "Generate hypothetical question", # 함수에 대한 설
        "parameters" : { # 함수의 매개변수 정의
            "type" : "object", # 매개변수의 타입을 객체로 지정
            "properties" : { "question" : { # 객체의 속성 정의, question 속성을 정의
                "type" : "array", "items" : {"type" : "string"}} # question의 타입을 배열로 지정, 배열의 요소 타입을 문자열로 지정
            }, "required" : ["question"] # 필수 매개변수로 question 지정
        }
    }
]

- 주어진 문서를 기반으로 3개의 가상질문 생성하는 프롬프트 템플릿 정의
- function 와 function_call 을 설정하여 가상 질문 생성 함수를 호출
- JsonKeyOutputFunctionParser 를 사용하여 생성된 가상 질문을 파싱하고, question 키에 해당하는 값을 추출

In [45]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.output_parsers.openai_functions import JsonKeyOutputFunctionsParser
from langchain_openai import ChatOpenAI

hypothetical_query_chain = (
    {"doc" : lambda x: x.page_content}
    # 아래 문서를 사용하여 답변할 수 있는 가상의 질문을 정확히 3개 생성하도록 요청. 이 숫자는 조정 가능
    | ChatPromptTemplate.from_template(
    "Generate a list of exactly 3 hypothetical questions that the below document could be used to answer. "
    "Potential users are those interested in the AI industry. Create questions that they would be interested in. "
    "Output should be written in Korean:\n\n{doc}"
    )
    | ChatOpenAI(max_retries=0, model="gpt-4o-mini").bind(
        functions= functions,
        function_call= {"name" : "hypothetical_question"}
    )
    # 출력에서 "questions" 키에 해당하는 값을 추출
    | JsonKeyOutputFunctionsParser(key_name="question")
)

In [46]:
# 출력은 생성한 3개의 가설 쿼리가 담겨 있음
# 주어진 문서에 대해 체인을 실행
hypothetical_query_chain.invoke(split_docs[33])

['삼성 가우스와 같은 온디바이스 생성 AI 모델이 향후 얼마나 많은 산업에 영향을 미칠 수 있을까?',
 '사용자의 개인정보 보호가 중요한 시대에 삼성 가우스의 개인정보 보호 기능이 기업의 신뢰성을 어떻게 향상시킬 수 있을까?',
 '삼성 가우스의 언어, 코드, 이미지 모델이 결합되어 제공하는 서비스의 장점은 무엇일까?']

In [47]:
# chain.batch 메서드를 사용하여 split_docs 데이터에 대해 동시에 여러개의 요청을 처리

# 문서 목록에 대해 가서러 질문을 배치 생성
hypothetical_questions = hypothetical_query_chain.batch(
    split_docs, {"max_concurrency": 10}
)

In [48]:
hypothetical_questions[33]

['삼성 가우스가 온디바이스에서 작동하는 경우, 이는 AI 산업에 어떤 영향을 미칠 것인가?',
 '온디바이스 AI 기술이 사용자 정보 보호에 기여할 수 있는 방법은 무엇인가?',
 '삼성전자 외에 다른 기업들도 비슷한 온디바이스 생성 AI 모델을 개발할 가능성이 있을까?']

In [49]:
# 이전에 진행했던 방식과 동일하게 생성한 가설 쿼리를 벡터저장소에 저장하는 과정

# 자식 청크를 인덱싱하는데 사용할 벡터 저장소
hypothetical_vectorstore = Chroma(
    collection_name="hypo-question",
    embedding_function= OpenAIEmbeddings()
)

# 부모 문서의 저장소 계층
store = InMemoryStore()

id_key = "doc_id"

# 검색기 ( 시작 시 비어 있음 )
retriever = MultiVectorRetriever(
    vectorstore= hypothetical_vectorstore,
    byte_store= store,
    id_key= id_key
)

doc_ids = [str(uuid.uuid4()) for _ in split_docs] # 문서 ID 생성

In [51]:
question_docs = []

# hypothetical_questions 저장
for i, question_list in enumerate(hypothetical_questions):
    question_docs.extend(
        # 질문 리스트의 각 질문에 대해 Document 객체를 생성하고, 메타데이터에 해당 질문의 문서 ID를 포함
        [Document(
            page_content=s,
            metadata= {id_key : doc_ids[i]}) for s in question_list]
    )

In [53]:
# 가설 쿼리를 문서에 추가하고, 원본 문서를 docstore에 추가

# hypothetical_questions 문서를 벡터 저장소에 추가
retriever.vectorstore.add_documents(question_docs)

# 문서 ID와 문서를 매핑하여 문서 저장소에 저장
retriever.docstore.mset(list(zip(doc_ids, split_docs)))

In [54]:
# 유사한 문서를 벡터 저장소에 검색
result_docs = hypothetical_vectorstore.similarity_search("삼성전자가 만든 생성형 AI의 이름은 ?")

In [55]:
# 유사도 검색 결과를 출력
for doc in result_docs:
    print(doc.page_content)
    print(doc.metadata)

삼성전자 외에 다른 기업들도 비슷한 온디바이스 생성 AI 모델을 개발할 가능성이 있을까?
{'doc_id': '2f9c2e1b-4846-4bf8-a642-512f2883dc07'}
삼성전자 외에 다른 기업들도 비슷한 온디바이스 생성 AI 모델을 개발할 가능성이 있을까?
{'doc_id': '2f9c2e1b-4846-4bf8-a642-512f2883dc07'}
삼성의 생성적 AI 연구 결과는 다른 기업들의 AI 발전에 어떤 기여를 할 수 있을까요?
{'doc_id': '682223b2-2bfd-4e6e-a666-7498c5a272df'}
삼성의 생성적 AI 연구 결과는 다른 기업들의 AI 발전에 어떤 기여를 할 수 있을까요?
{'doc_id': '682223b2-2bfd-4e6e-a666-7498c5a272df'}


In [56]:
# 관련된 문서를 검색하여 가져옵니다.
retrieved_docs = retriever.invoke(result_docs[1].page_content)

# 검색된 문서를 출력합니다.
for doc in retrieved_docs:
    print(doc.page_content)

SPRi AI Brief |  
2023-12월호
10
삼성전자, 자체 개발 생성 AI ‘삼성 가우스’ 공개
n 삼성전자가 온디바이스에서 작동 가능하며 언어, 코드, 이미지의 3개 모델로 구성된 자체 개발 생성 
AI 모델 ‘삼성 가우스’를 공개
n 삼성전자는 삼성 가우스를 다양한 제품에 단계적으로 탑재할 계획으로, 온디바이스 작동이 가능한 
삼성 가우스는 외부로 사용자 정보가 유출될 위험이 없다는 장점을 보유
KEY Contents
£ 언어, 코드, 이미지의 3개 모델로 구성된 삼성 가우스, 온디바이스 작동 지원
n 삼성전자가 2023년 11월 8일 열린 ‘삼성 AI 포럼 2023’ 행사에서 자체 개발한 생성 AI 모델 
‘삼성 가우스’를 최초 공개
∙정규분포 이론을 정립한 천재 수학자 가우스(Gauss)의 이름을 본뜬 삼성 가우스는 다양한 상황에 
최적화된 크기의 모델 선택이 가능
∙삼성 가우스는 라이선스나 개인정보를 침해하지 않는 안전한 데이터를 통해 학습되었으며, 
온디바이스에서 작동하도록 설계되어 외부로 사용자의 정보가 유출되지 않는 장점을 보유
∙삼성전자는 삼성 가우스를 활용한 온디바이스 AI 기술도 소개했으며, 생성 AI 모델을 다양한 제품에
▹ 코히어, 데이터 투명성 확보를 위한 데이터 출처 탐색기 공개  ······································· 8
   ▹ 알리바바 클라우드, 최신 LLM ‘통이치엔원 2.0’ 공개 ······················································ 9
   ▹ 삼성전자, 자체 개발 생성 AI ‘삼성 가우스’ 공개 ··························································· 10
   ▹ 구글, 앤스로픽에 20억 달러 투자로 생성 AI 협력 강화 ················································ 11
   ▹ IDC, 2027년 AI 소프트웨어 매출 2,500억 달러 돌